# App assets

Builds the small files the Streamlit app reads (`app/assets/`). The app never loads a model: every
number it shows is computed here, once, from the same split and model files as the evaluation
notebooks.

**How to run it (`MACHINE`):**
- **`"all"`** (the normal way, on bugatti): one run does everything for all three models. TabPFN
  is scored on this split's test set on the GPU, so all three models' scores line up row by row
  by construction.
- **`"laptop"` then `"bugatti"`** (fallback): the laptop pass does the checks, the grid, the
  applicants and the per-applicant steps for logistic regression and XGBoost, reading TabPFN's
  scores from its exported file; the bugatti pass adds TabPFN's per-applicant steps and the XGBoost
  refits. Each pass only replaces its own models' rows in the asset files.

**Inputs:** `data/processed/{train,test}.parquet`, `models/{logreg,xgboost,tabpfn}_model.joblib`,
and the summaries the evaluation notebooks left in `outputs/evaluation/`. Only the `"laptop"` pass
reads `data/tabpfn-results/test_predictions_with_sensitive.parquet`, which must come from this
same copy of the split.

In [1]:
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics import brier_score_loss, roc_auc_score

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))
import common_metrics as cm  # also puts src/ on the path, so the saved models unpickle

import logreg_model
import tabpfn_model
import xgboost_model

MODULES = {"logreg": logreg_model, "xgboost": xgboost_model, "tabpfn": tabpfn_model}

In [2]:
MACHINE = "all"      # "all", or "laptop" then "bugatti", see above
DRY_RUN = False      # True: fit quick stand-in models when a model file is missing (to test the app only)

ASSETS = REPO / "app" / "assets"
TEST_PATH = cm.DATA_DIR / "test.parquet"
TRAIN_PATH = cm.DATA_DIR / "train.parquet"
TABPFN_PREDICTIONS = cm.PRECOMPUTED_PREDICTIONS["tabpfn"]
TABPFN_SCORES_CACHE = REPO / "data" / "app-run" / "tabpfn_test_scores.parquet"  # "all" only; saves a re-run

RANDOM_STATE = 42
MARGIN, LGD = 0.03, 0.35                 # as xgboost_model.ipynb and financial.ipynb
THRESHOLD = cm.TEAM_THRESHOLD            # LGD / (MARGIN + LGD)
THRESHOLDS = np.round(np.arange(0.300, 0.9995, 0.001), 3)

# Each protected attribute is compared against a reference group. "Not available" answers and
# groups under MIN_GROUP_N test rows are left out: their ratios are mostly noise.
ATTRIBUTES = {
    "derived_race": "White",
    "derived_ethnicity": "Not Hispanic or Latino",
    "derived_sex": "Male",
    "applicant_age": "35-44",
}
MIN_GROUP_N = 1_000
NOT_A_GROUP = {"Race Not Available", "Ethnicity Not Available", "Sex Not Available", "Free Form Text Only"}

N_PER_GROUP = 30     # sampled applicants per race group, half approved and half denied by the lender
N_EXTRA = 45         # plus this many where the models disagree, and this many close to the threshold
N_REFERENCE = 20     # typical applicants a feature is swapped with to measure its effect
N_REFITS, REFIT_ROWS = 10, 500_000
WHATIF_FEATURES = ["income", "loan_amount", "combined_loan_to_value_ratio", "debt_to_income_ratio"]
WHATIF_QUANTILES = np.linspace(0.05, 0.95, 10)

PER_APPLICANT = {
    "all": {"reasons": ["logreg", "xgboost", "tabpfn"], "whatif": ["logreg", "xgboost", "tabpfn"],
            "refits": ["logreg", "xgboost", "tabpfn"]},
    "laptop": {"reasons": ["logreg", "xgboost"], "whatif": ["logreg", "xgboost"], "refits": ["logreg"]},
    "bugatti": {"reasons": ["tabpfn"], "whatif": ["tabpfn"], "refits": ["tabpfn", "xgboost"]},
}[MACHINE]

SHARED_STEPS = MACHINE in ("all", "laptop")  # full-test scores, grid, applicants, summary
ASSETS.mkdir(parents=True, exist_ok=True)
print(f"{MACHINE} pass{' (DRY RUN)' if DRY_RUN else ''} -> {ASSETS}")

all pass -> /users/eleves-b/2025/laikhuram.korouhanba-khuman/ISAF_app_run/app/assets


## 1. Data and models

`RAW` is the 30 raw columns XGBoost and TabPFN read (logistic regression uses 12 of them). The
protected columns come along for the fairness grid only; no model sees them.

In [3]:
RAW = xgboost_model.RAW_COLUMNS
AUDIT = list(ATTRIBUTES)
assert tabpfn_model.RAW_COLUMNS == RAW and set(logreg_model.FEATURES) <= set(RAW)

test = pd.read_parquet(TEST_PATH, columns=RAW + AUDIT + ["target"])
test.index.name = "row_nr"
y = test["target"].to_numpy()
amount = test["loan_amount"].to_numpy(dtype=float)
assert len(test) == 1_368_986, f"test has {len(test):,} rows, not the 1,368,986 the evaluation notebooks used"
print(f"test: {len(test):,} rows, lender approval rate {y.mean():.3f}")

_train = None


def train_rows(n: int, seed: int, replace: bool = False) -> pd.DataFrame:
    # Random training rows (a bootstrap sample when replace=True), read once and kept in polars.
    global _train
    if _train is None:
        _train = pl.read_parquet(TRAIN_PATH, columns=RAW + AUDIT + ["target"])
    idx = np.random.default_rng(seed).choice(_train.height, size=n, replace=replace)
    return _train[idx].to_pandas()


def load_model(name: str):
    path = cm.ARTIFACTS_DIR / f"{name}_model.joblib"
    if path.exists():
        return joblib.load(path)
    if not DRY_RUN:
        raise FileNotFoundError(f"{path} is missing, copy it in from the team's files")
    print(f"  DRY RUN: no {path.name}, fitting a stand-in on 200k training rows")
    sample = train_rows(200_000, RANDOM_STATE)
    kwargs = {"params": {**xgboost_model.load_params(), "n_estimators": 200}} if name == "xgboost" else {}
    return MODULES[name].fit(sample, sample["target"], **kwargs)


def score(name: str, model, X: pd.DataFrame, batch: int = 200_000) -> np.ndarray:
    return np.concatenate([MODULES[name].predict_proba(model, X.iloc[i:i + batch])[:, 1]
                           for i in range(0, len(X), batch)])


# Refits train their own models, so each pass only loads the saved models it scores with.
models = {name: load_model(name) for name in
          {"all": ["logreg", "xgboost", "tabpfn"], "laptop": ["logreg", "xgboost"], "bugatti": ["tabpfn"]}[MACHINE]}
print("models loaded:", list(models))

test: 1,368,986 rows, lender approval rate 0.775
models loaded: ['logreg', 'xgboost', 'tabpfn']


/users/eleves-b/2025/laikhuram.korouhanba-khuman/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/pickle.py:1718: UserWarning: [00:33:32] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)
/users/eleves-b/2025/laikhuram.korouhanba-khuman/isaf-venv/lib/python3.11/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
htt

## 2. Scores on the full test set

Logistic regression and XGBoost are scored here. TabPFN is scored here too with `MACHINE = "all"`
(about 8 minutes on the GPU, cached in `data/app-run/`); with `"laptop"` its scores come from the
file `tabpfn.ipynb` exported, which must line up with `test.parquet` row by row, otherwise every
comparison below would pair different applicants.

In [4]:
if SHARED_STEPS:
    scores = {name: score(name, models[name], test) for name in ["logreg", "xgboost"]}
    if MACHINE == "all":
        if TABPFN_SCORES_CACHE.exists():
            cached = pd.read_parquet(TABPFN_SCORES_CACHE)
            assert (cached["target"].to_numpy() == y).all(), "cached TabPFN scores are from another split"
        else:
            cached = pd.DataFrame({"target": y, "score": score("tabpfn", models["tabpfn"], test)}, index=test.index)
            TABPFN_SCORES_CACHE.parent.mkdir(parents=True, exist_ok=True)
            cached.to_parquet(TABPFN_SCORES_CACHE)
        scores["tabpfn"] = cached["score"].to_numpy()
    elif TABPFN_PREDICTIONS.exists():
        pred = pd.read_parquet(TABPFN_PREDICTIONS).reindex(test.index)
        assert pred["true_target"].notna().all() and (pred["true_target"].to_numpy() == y).all(), \
            "TabPFN predictions don't line up with test.parquet: they were made on a different split"
        scores["tabpfn"] = pred["predicted_probability"].to_numpy()
    else:
        print(f"no {TABPFN_PREDICTIONS.name}, TabPFN is left out of this pass")

    def pnl(approve):
        return MARGIN * amount[approve & (y == 1)].sum() - LGD * amount[approve & (y == 0)].sum()

    summary = pd.DataFrame({
        name: {"auc": roc_auc_score(y, p), "brier": brier_score_loss(y, p),
               "approval_rate": (p >= THRESHOLD).mean(), "pnl": pnl(p >= THRESHOLD)}
        for name, p in scores.items()
    }).T
    print("should match financial.ipynb at 0.921 (P&L $bn: xgboost 4.700, logreg 1.376; "
          "tabpfn about 3.976, its scores shift slightly with the batch)")
    display(summary.assign(pnl_bn=summary["pnl"] / 1e9).drop(columns="pnl").round(3))

should match financial.ipynb at 0.921 (P&L $bn: xgboost 4.700, logreg 1.376; tabpfn about 3.976, its scores shift slightly with the batch)


,auc,brier,approval_rate,pnl_bn
logreg,0.765,0.138,0.205,1.414
xgboost,0.903,0.081,0.473,4.700
tabpfn,0.876,0.095,0.395,3.973


## 3. Threshold grid

For every model, threshold (0.300 to 0.999) and group: how many applicants the model approves,
how many of those the lender also approved, and the loan amounts behind them. The app turns these
counts into approval rates, TPR, impact ratios and P&L for any margin, LGD or default assumption
without touching the 1.4M rows again.

Scores are binned to 0.001, the same step as the thresholds, so `p >= t` is exact on this grid.

In [5]:
CUTS = np.round(THRESHOLDS * 1000).astype(int)


def grid_rows(name: str, p: np.ndarray) -> pd.DataFrame:
    bins = np.minimum((p * 1000).astype(int), 1000)
    frames = []
    for attr in ["all"] + AUDIT:
        group = np.full(len(p), "all", dtype=object) if attr == "all" else test[attr].to_numpy(dtype=object)
        counts = pd.DataFrame({"group": group, "bin": bins, "n": 1, "n_pos": y,
                               "amt_pos": amount * y, "amt_neg": amount * (1 - y)}).groupby(["group", "bin"]).sum()
        for g, by_bin in counts.groupby(level="group"):
            if attr != "all" and (g in NOT_A_GROUP or by_bin["n"].sum() < MIN_GROUP_N):
                continue
            by_bin = by_bin.droplevel("group").reindex(range(1001), fill_value=0)
            at = by_bin[::-1].cumsum()[::-1].loc[CUTS]  # totals over every bin >= the threshold
            frames.append(pd.DataFrame({
                "model": name, "attribute": attr, "group": g, "threshold": THRESHOLDS,
                "n": by_bin["n"].sum(), "n_pos": by_bin["n_pos"].sum(),
                "approved": at["n"].to_numpy(), "approved_pos": at["n_pos"].to_numpy(),
                "amt_approved_pos": at["amt_pos"].to_numpy(), "amt_approved_neg": at["amt_neg"].to_numpy(),
            }))
    return pd.concat(frames, ignore_index=True)


if SHARED_STEPS:
    grid = pd.concat([grid_rows(name, p) for name, p in scores.items()], ignore_index=True)
    grid.to_parquet(ASSETS / "grid.parquet", index=False)
    print(f"grid: {len(grid):,} rows; groups kept:")
    display(grid.query("model == 'logreg' and threshold == 0.5").groupby("attribute")["group"].apply(list))

grid: 44,100 rows; groups kept:


attribute
all                                                              [all]
applicant_age            [25-34, 35-44, 45-54, 55-64, 65-74, <25, >74]
derived_ethnicity    [Hispanic or Latino, Joint, Not Hispanic or La...
derived_race         [2 or more minority races, American Indian or ...
derived_sex                                      [Female, Joint, Male]
Name: group, dtype: object

## 4. Applicants shown in the app

About 300 test applicants: for every race group, some the lender approved and some it denied,
plus cases where the models disagree at the team threshold and cases close to it, where
explanations and stability matter most.

In [6]:
if SHARED_STEPS:
    rng = np.random.default_rng(RANDOM_STATE)
    all_scores = pd.DataFrame(scores, index=test.index)
    picked = {}
    race_counts = test["derived_race"].value_counts()
    for race in [g for g, n in race_counts.items() if g not in NOT_A_GROUP and n >= MIN_GROUP_N]:
        for outcome, word in [(1, "approved"), (0, "denied")]:
            pool = test.index[(test["derived_race"] == race).to_numpy() & (y == outcome)]
            for r in rng.choice(pool, size=min(N_PER_GROUP // 2, len(pool)), replace=False):
                picked.setdefault(r, f"{race}, lender {word}")
    disagree = all_scores.index[(all_scores >= THRESHOLD).nunique(axis=1) > 1]
    near = all_scores.index[(all_scores - THRESHOLD).abs().min(axis=1) < 0.02]
    for pool, why in [(disagree, "models disagree"), (near, "close to the threshold")]:
        pool = pool.difference(list(picked))
        for r in rng.choice(pool, size=min(N_EXTRA, len(pool)), replace=False):
            picked[r] = why

    applicants = test.loc[sorted(picked)].copy()
    applicants["picked_because"] = [picked[r] for r in applicants.index]
    for name in scores:
        applicants[f"score_{name}"] = all_scores.loc[applicants.index, name]
    applicants.reset_index().to_parquet(ASSETS / "applicants.parquet", index=False)
    print(f"{len(applicants)} applicants")
    display(applicants["picked_because"].value_counts())

300 applicants


picked_because
models disagree                                               45
close to the threshold                                        45
2 or more minority races, lender approved                     15
Native Hawaiian or Other Pacific Islander, lender denied      15
American Indian or Alaska Native, lender denied               15
Asian, lender denied                                          15
2 or more minority races, lender denied                       15
American Indian or Alaska Native, lender approved             15
Asian, lender approved                                        15
Native Hawaiian or Other Pacific Islander, lender approved    15
White, lender denied                                          15
Black or African American, lender approved                    15
White, lender approved                                        15
Joint, lender denied                                          15
Joint, lender approved                                        15
Black or A

## 5. Per-applicant steps

Every model is explained and tested on the same applicants, the same reference rows and the same
what-if values.

- **Reasons.** A feature's effect is how much this applicant's approval probability changes when
  that feature alone is swapped for the values of `N_REFERENCE` typical applicants. Negative
  means the feature pushed towards denial; the most negative ones are the principal reasons an
  adverse action notice would give. The same method is used for all three models, so their
  explanations are comparable (logistic regression's 18 unused features come out at 0).
- **What-if.** Each applicant rescored with one input moved across its test-set range (5th to
  95th percentile), everything else unchanged.
- **Refits.** Each applicant scored by `N_REFITS` models retrained on bootstrap samples of
  `REFIT_ROWS` training rows (for TabPFN, a new context each time).

In [7]:
applicants = pd.read_parquet(ASSETS / "applicants.parquet").set_index("row_nr")
X_app = applicants[RAW]
reference = test.sample(N_REFERENCE, random_state=RANDOM_STATE)[RAW]
whatif_values = {f: np.unique(test[f].quantile(WHATIF_QUANTILES).to_numpy()) for f in WHATIF_FEATURES}


def save_rows(file: str, frame: pd.DataFrame) -> None:
    # Replaces only the rows of the models in `frame`, keeping the other pass's rows.
    path = ASSETS / f"{file}.parquet"
    if path.exists():
        old = pd.read_parquet(path)
        frame = pd.concat([old[~old["model"].isin(frame["model"].unique())], frame], ignore_index=True)
    frame.to_parquet(path, index=False)


def feature_effects(name: str) -> pd.DataFrame:
    base = score(name, models[name], X_app)
    frames = []
    for f in RAW:
        swapped = X_app.loc[X_app.index.repeat(len(reference))].copy()
        swapped[f] = np.tile(reference[f].to_numpy(), len(X_app))
        p = score(name, models[name], swapped).reshape(len(X_app), len(reference)).mean(axis=1)
        frames.append(pd.DataFrame({"row_nr": X_app.index, "model": name, "feature": f, "effect": base - p}))
    return pd.concat(frames, ignore_index=True)


def whatif(name: str) -> pd.DataFrame:
    frames = []
    for f, values in whatif_values.items():
        moved = X_app.loc[X_app.index.repeat(len(values))].copy()
        moved[f] = np.tile(values, len(X_app))
        frames.append(pd.DataFrame({"row_nr": moved.index, "model": name, "feature": f,
                                    "value": moved[f].to_numpy(dtype=float), "score": score(name, models[name], moved)}))
    return pd.concat(frames, ignore_index=True)


def refits(name: str) -> pd.DataFrame:
    frames = []
    for i in range(N_REFITS):
        boot = train_rows(REFIT_ROWS, seed=RANDOM_STATE + i, replace=True)
        refit = MODULES[name].fit(boot, boot["target"])
        frames.append(pd.DataFrame({"row_nr": X_app.index, "model": name, "refit": i,
                                    "score": score(name, refit, X_app)}))
        print(f"  {name} refit {i + 1}/{N_REFITS}")
    return pd.concat(frames, ignore_index=True)


for step, fn in [("reasons", feature_effects), ("whatif", whatif), ("refits", refits)]:
    for name in PER_APPLICANT[step]:
        save_rows(step, fn(name))
        print(f"{step}: {name} done")

reasons: logreg done
reasons: xgboost done
reasons: tabpfn done
whatif: logreg done
whatif: xgboost done
whatif: tabpfn done


/users/eleves-b/2025/laikhuram.korouhanba-khuman/isaf-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  logreg refit 1/10


/users/eleves-b/2025/laikhuram.korouhanba-khuman/isaf-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  logreg refit 2/10


...

  logreg refit 8/10


/users/eleves-b/2025/laikhuram.korouhanba-khuman/isaf-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  logreg refit 9/10


/users/eleves-b/2025/laikhuram.korouhanba-khuman/isaf-venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  logreg refit 10/10
refits: logreg done
  xgboost refit 1/10
  xgboost refit 2/10
  xgboost refit 3/10
  xgboost refit 4/10
  xgboost refit 5/10
  xgboost refit 6/10
  xgboost refit 7/10
  xgboost refit 8/10
  xgboost refit 9/10
  xgboost refit 10/10
refits: xgboost done
  tabpfn refit 1/10
  tabpfn refit 2/10
  tabpfn refit 3/10
  tabpfn refit 4/10
  tabpfn refit 5/10
  tabpfn refit 6/10
  tabpfn refit 7/10
  tabpfn refit 8/10
  tabpfn refit 9/10
  tabpfn refit 10/10
refits: tabpfn done


## 6. Summary for the app

`meta.json` holds the settings above, the headline numbers per model, and the totals the app
needs for P&L. The evaluation notebooks' summary tables and a few of their charts are copied
as they are.

In [8]:
if SHARED_STEPS:
    meta = {
        "generated_at": datetime.now().isoformat(timespec="minutes"),
        "dry_run": DRY_RUN,
        "margin": MARGIN, "lgd": LGD, "threshold": THRESHOLD,
        "attributes": ATTRIBUTES, "min_group_n": MIN_GROUP_N,
        "n_test": len(test), "lender_approval_rate": float(y.mean()),
        "amt_pos_total": float(amount[y == 1].sum()), "amt_neg_total": float(amount[y == 0].sum()),
        "models": {name: {k: float(v) for k, v in row.items()} for name, row in summary.iterrows()},
        "reference_rows": reference.index.tolist(),
        "whatif_values": {f: v.tolist() for f, v in whatif_values.items()},
    }
    (ASSETS / "meta.json").write_text(json.dumps(meta, indent=2))

    framework_dir = ASSETS / "framework"
    framework_dir.mkdir(exist_ok=True)
    for csv in sorted(cm.OUTPUT_DIR.rglob("*.csv")):
        if csv.stat().st_size < 1e6:
            shutil.copy(csv, framework_dir / csv.name)

    charts_dir = ASSETS / "charts"
    charts_dir.mkdir(exist_ok=True)
    for chart in ["fairness/fairness_disparate_impact_pair.png", "fairness/fairness_equalized_odds_pair.png",
                  "interpretability/interpretability_shap_comparison.png",
                  "interpretability/interpretability_surrogate_fidelity_comparison.png",
                  "stability/stability_bootstrap_distributions.png", "financial/financial_pnl_test.png"]:
        if (cm.OUTPUT_DIR / chart).exists():
            shutil.copy(cm.OUTPUT_DIR / chart, charts_dir / Path(chart).name)
else:
    meta = json.loads((ASSETS / "meta.json").read_text())
    assert meta["reference_rows"] == reference.index.tolist(), "different reference rows than the laptop pass"

In [9]:
for path in sorted(ASSETS.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(ASSETS)):60s} {path.stat().st_size / 1e6:6.2f} MB")
for file in ["reasons", "whatif", "refits"]:
    if (ASSETS / f"{file}.parquet").exists():
        print(file, sorted(pd.read_parquet(ASSETS / f"{file}.parquet")["model"].unique()))

.gitignore                                                     0.00 MB
applicants.parquet                                             0.04 MB
charts/fairness_disparate_impact_pair.png                      0.07 MB
charts/fairness_equalized_odds_pair.png                        0.08 MB
charts/financial_pnl_test.png                                  0.04 MB
charts/interpretability_shap_comparison.png                    0.13 MB
charts/interpretability_surrogate_fidelity_comparison.png      0.04 MB
charts/stability_bootstrap_distributions.png                   0.08 MB
framework/fairness_detail.csv                                  0.00 MB
framework/fairness_summary.csv                                 0.00 MB
framework/interpretability_summary.csv                         0.00 MB
grid.parquet                                                   0.93 MB
meta.json                                                      0.00 MB
reasons.parquet                                                0.16 MB
refits